# Candidate Recommendation (Employer) — Embedding Baseline

**Goal:** Given a **job posting** (title + cleaned description), recommend **top-K candidate resumes** using **Sentence-BERT embeddings + cosine similarity**.

This is a **baseline**: only semantic similarity (no taxonomy/skills reranking here).

**Inputs**
- `./datasets/it_jobs.xlsx`
- `./datasets/UpdatedResumeDataSet.csv`

**Outputs**
- `./outputs/candrec_emb/baseline_topk.csv`
- cached resume embeddings: `./outputs/candrec_emb/resume_embeddings_all-MiniLM-L6-v2.npy`


In [8]:

import os, re, json, math
import numpy as np
import pandas as pd

from pathlib import Path

DATA_DIR = Path("./datasets")
OUT_DIR  = Path("./outputs/candrec_emb")
OUT_DIR.mkdir(parents=True, exist_ok=True)

JOBS_XLSX = DATA_DIR / "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/jobs_merged_it.xlsx"
CVS_CSV   = DATA_DIR / "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/UpdatedResumeDataSet.csv"

assert JOBS_XLSX.exists(), f"Missing: {JOBS_XLSX.resolve()}"
assert CVS_CSV.exists(),   f"Missing: {CVS_CSV.resolve()}"

jobs = pd.read_excel(JOBS_XLSX)
cvs  = pd.read_csv(CVS_CSV)

print("jobs:", jobs.shape, "| cvs:", cvs.shape)
print("jobs columns:", list(jobs.columns))
print("cvs columns:", list(cvs.columns))


jobs: (89277, 5) | cvs: (962, 2)
jobs columns: ['title', 'description', 'company', 'location', 'source']
cvs columns: ['Category', 'Resume']


In [9]:

# --- Clean & build text fields ---

def norm_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Jobs
jobs["title"] = jobs["title"].map(norm_text)
jobs["cleaned_description"] = jobs.get("cleaned_description", pd.Series("", index=jobs.index))
jobs["cleaned_description"] = jobs["cleaned_description"].map(norm_text)


jobs["job_text"] = (jobs["title"] + ". " + jobs["cleaned_description"]).map(norm_text)

# CVs
cvs["Category"] = cvs["Category"].map(norm_text)
cvs["Resume"]   = cvs["Resume"].map(norm_text)

# Optional: quick sanity
print("Empty job_text:", (jobs["job_text"].str.len()==0).sum())
print("Empty resume:", (cvs["Resume"].str.len()==0).sum())


Empty job_text: 0
Empty resume: 0


In [10]:

# --- Embedding model ---
# If you don't have sentence-transformers yet:
#   pip install -U sentence-transformers

from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

print("Loaded:", MODEL_NAME)


Loaded: sentence-transformers/all-MiniLM-L6-v2


In [11]:

# --- Cache resume embeddings (so you don't re-encode every run) ---

emb_path = OUT_DIR / f"resume_embeddings_{MODEL_NAME.split('/')[-1]}.npy"

if emb_path.exists():
    resume_emb = np.load(emb_path)
    print("Loaded cached resume embeddings:", resume_emb.shape, "from", emb_path)
else:
    resume_emb = model.encode(
        cvs["Resume"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    np.save(emb_path, resume_emb)
    print("Saved resume embeddings:", resume_emb.shape, "to", emb_path)


Loaded cached resume embeddings: (962, 384) from outputs/candrec_emb/resume_embeddings_all-MiniLM-L6-v2.npy


In [13]:

# --- Recommend candidates for a chosen job ---

from numpy.linalg import norm

def topk_for_job(job_index: int, k: int = 10):
    job_row = jobs.iloc[job_index]
    job_vec = model.encode([job_row["job_text"]], normalize_embeddings=True)[0]

    # cosine similarity (normalized embeddings => dot product)
    sims = resume_emb @ job_vec
    top_idx = np.argsort(-sims)[:k]

    out = pd.DataFrame({
        "cv_index": top_idx,
        "cv_category": cvs.loc[top_idx, "Category"].values,
        "score": sims[top_idx],
        "resume_snippet": cvs.loc[top_idx, "Resume"].str.slice(0, 220).values,
    })
    out.insert(0, "job_id", job_row.get("id", job_index))
    out.insert(1, "job_title", job_row["title"])
    return out

# Pick a job to test (change this)
JOB_INDEX = 0
K = 10

res = topk_for_job(JOB_INDEX, K)
res


,job_id,job_title,cv_index,cv_category,score,resume_snippet
0,0,Support Technologist II 2024-02216,418,Business Analyst,0.471424,"Technical Skills Application Servers: IIS 6.0,..."
1,0,Support Technologist II 2024-02216,404,Business Analyst,0.471424,"Technical Skills Application Servers: IIS 6.0,..."
2,0,Support Technologist II 2024-02216,5,Data Science,0.462661,"SKILLS C Basics, IOT, Python, MATLAB, Data Sci..."
3,0,Support Technologist II 2024-02216,35,Data Science,0.462661,"SKILLS C Basics, IOT, Python, MATLAB, Data Sci..."
4,0,Support Technologist II 2024-02216,25,Data Science,0.462661,"SKILLS C Basics, IOT, Python, MATLAB, Data Sci..."
5,0,Support Technologist II 2024-02216,15,Data Science,0.462661,"SKILLS C Basics, IOT, Python, MATLAB, Data Sci..."
6,0,Support Technologist II 2024-02216,37,Data Science,0.460307,Education Details B.Tech Rayat and Bahra Insti...
7,0,Support Technologist II 2024-02216,7,Data Science,0.460307,Education Details B.Tech Rayat and Bahra Insti...
8,0,Support Technologist II 2024-02216,27,Data Science,0.460307,Education Details B.Tech Rayat and Bahra Insti...
9,0,Support Technologist II 2024-02216,17,Data Science,0.460307,Education Details B.Tech Rayat and Bahra Insti...


### Đánh giá chất lượng Candidate Recommendation (Embedding Baseline)

| Tiêu chí | Đánh giá | Nhận xét |
|--------|---------|----------|
| Semantic Score | Ổn | Score ~0.46 cho thấy mức độ tương đồng ngữ nghĩa trung bình–khá |
| Phân biệt ứng viên | Kém | Điểm số giữa các ứng viên gần như tương đương |
| Phù hợp vai trò | Yếu | Xuất hiện nhiều CV Data Science cho vị trí IT Support |
| Explainability | Thấp | Khó giải thích lý do gợi ý nếu chỉ dựa vào embedding |
| Giá trị nghiên cứu | Cao | Phù hợp làm baseline để so sánh với mô hình improved |


In [7]:

# --- Export baseline results ---
out_path = OUT_DIR / "baseline_topk.csv"
res.to_csv(out_path, index=False)
print("Saved:", out_path.resolve())


Saved: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebooks/employer/EBD/outputs/candrec_emb/baseline_topk.csv
